# Math Reasoning LLM - Operational Launcher

This notebook serves as the operational launcher for training, evaluation, and data generation.
It works seamlessly both **in Google Colab** (with T4/G4/A100 GPUs) and **in Antigravity IDE / local Jupyter**.

> **Architecture Rules:**
> 1. **No Training Logic in Notebook Cells:** All post-training, data synthesis, and evaluation runs inside modular scripts in `scripts/`.
> 2. **Persistence on Google Drive (Colab mode):** Artifacts (`checkpoints/`, `outputs/`, `logs/`, `data/`) are saved directly to `My Drive/math-reasoning-llm/`.
> 3. **High-Speed Model Caching:** Hugging Face model cache remains on fast local SSD disk (`/content/hf_cache` on Colab, or local `.cache/`), NEVER on Google Drive.
> 4. **Zero-Secret Commits:** Hugging Face token is loaded directly from environment or Secrets (`HF_TOKEN`).

## Step 1: Environment Detection & Persistence Setup
Detects whether running in Google Colab (mounts Google Drive) or local workstation, and sets the root directory.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure current working directory is the repository root, not notebooks/
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print(f"Active working directory: {Path.cwd()}")

# Check if running inside Google Colab
IS_COLAB = "google.colab" in sys.modules
try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("Running in Google Colab. Mounting Google Drive...")
    drive.mount('/content/drive')
    DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/math-reasoning-llm')
    for subfolder in ['checkpoints', 'outputs', 'logs', 'data']:
        (DRIVE_PROJECT_ROOT / subfolder).mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted successfully at: {DRIVE_PROJECT_ROOT}")
else:
    print("Running in local/IDE environment. Using local workspace storage.")
    for subfolder in ['checkpoints', 'outputs', 'logs', 'data']:
        Path(subfolder).mkdir(parents=True, exist_ok=True)
    print("Local directories verified.")

## Step 2: Repository Branch Synchronization
Ensures latest project code is checked out.

In [ ]:
REPO_URL = "https://github.com/NashLucas/math-reasoning-llm.git"
REPO_DIR = "/content/math-reasoning-llm"
BRANCH = "phase-2-generation-and-baseline"

if IS_COLAB:
    if not os.path.exists(REPO_DIR):
        print(f"Cloning {REPO_URL}...")
        !git clone {REPO_URL} {REPO_DIR}
        %cd {REPO_DIR}
        !git checkout {BRANCH}
    else:
        print(f"Updating repository in {REPO_DIR}...")
        %cd {REPO_DIR}
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    %cd {REPO_DIR}
else:
    print("Local workspace already loaded.")

!git status

## Step 3: Hardware Diagnostics & Environment Configuration
Inspects GPU hardware (CUDA, VRAM) and configures HF cache.

In [ ]:
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

# 1. HF Token
if IS_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print("HF token loaded from Colab Secrets.")
    except Exception:
        print("NOTE: HF_TOKEN not set in Colab Secrets (optional for public models like Qwen / DeepSeek-Math).")
    os.environ['HF_HOME'] = '/content/hf_cache'
    os.environ['APP_PROFILE'] = 'colab'
    os.makedirs('/content/hf_cache', exist_ok=True)

# 2. Diagnostics
print("=" * 60)
print("HARDWARE DIAGNOSTICS")
print("=" * 60)
if HAS_TORCH and torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Detected: {gpu_name}")
    print(f"VRAM: {vram_gb:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"PyTorch Version: {torch.__version__}")
else:
    print("CPU Runtime detected (No CUDA GPU active).")
    if IS_COLAB:
        print("To enable GPU on Colab: Runtime -> Change runtime type -> T4 GPU.")
print("=" * 60)

## Step 4: Install Dependencies

In [ ]:
if IS_COLAB:
    !pip install -r requirements.txt
else:
    print("Dependencies managed by local environment.")

## Step 5: Verify Configurations & Stage Paths

In [ ]:
!python src/config.py

## Step 5b: Prepare GSM8K Splits on the Profile Data Directory

Generated data (`raw/`, `processed/`, `safety/`) lives in the profile data directory: `data/` locally, `My Drive/math-reasoning-llm/data/` on Colab, so it survives a disconnect.
Data preparation is deterministic (bit-identical splits), so this step is skipped when the processed splits already exist on Drive.
Evaluation never falls back to the 20-problem sample set on a real run: a missing split stops the run instead.

In [ ]:
from src.config import ConfigManager

_processed_val = ConfigManager().data_path('processed', 'val', 'val.parquet')
if _processed_val.exists():
    print(f'Processed splits found: {_processed_val.parent.parent}')
else:
    print(f'Processed splits missing at {_processed_val.parent.parent}; preparing data...')
    !python scripts/prepare_data.py


## Step 5c: Day 5 — vLLM Check (run once per runtime type, e.g. on G4)

Installs `requirements-vllm.txt` and runs a tiny generation with the dev model.
Writes the verdict to `outputs/env_checks/vllm_check.json` (Drive under colab); log it in `EXPERIMENTS.md`.
If the verdict is `use_hf_fallback`, apply the vLLM row of the plan's fallback rules.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    !pip install -q -r requirements-vllm.txt -c requirements.txt
    !python scripts/check_vllm.py
else:
    print('vLLM needs a CUDA GPU; skipping (CPU / local).')


## Step 5d: Phase 0 Gate — Training Resumes After a Disconnect

Trains a tiny GPT-2 with the Hugging Face Trainer for 50 steps (checkpoint every 10) into `checkpoints/resume_test` (Drive under colab). A CPU runtime is enough.

**Real disconnect test (the gate):**
1. Run cell **1)**. It starts a fresh test and prints `step N/50` and `saved checkpoint-N` as it goes.
2. After `saved checkpoint-10` and **before step 50**, use **Runtime → Disconnect and delete runtime** (not "Restart session": the machine and its disk must be gone).
3. **Connect** to a new runtime, re-run Steps 1–4, then run cell **2)**. It must print `Resuming from checkpoint-N` with N > 0 and finish at step 50.
4. Run cell **3)**. It must print `PHASE 0 RESUME GATE: PASSED`, including `[ok] Resumed on a different machine`.

Never run cell 1) after the disconnect: `--fresh` deletes the checkpoints. Re-running cell 2) after training finished does nothing.
Cell **4)** is an optional quick check: a hard process kill and resume on the same machine (`[info]` for the machine check).

In [ ]:
# 1) FIRST RUN ONLY: start a fresh test, then disconnect and delete the runtime mid-run
!python scripts/test_resume_training.py --mode run --fresh --step_delay 3.0


In [ ]:
# 2) AFTER RECONNECTING to a new runtime: resume from the last checkpoint on Drive
!python scripts/test_resume_training.py --mode run --step_delay 0.3


In [ ]:
# 3) Verify the gate
!python scripts/test_resume_training.py --mode verify


In [ ]:
# 4) Optional quick check: hard process kill + resume on the same machine (starts fresh)
!python scripts/test_resume_training.py --mode auto


## Step 6: Day 11 Verification — Development Model Generation & Chunk Resumption

- On GPU (Colab T4): runs with `--use_real_model` to load `Qwen/Qwen2.5-1.5B-Instruct` on GPU.
- On CPU (Local): runs without `--use_real_model` to test stop rules and resumption with the mock generator.

In [ ]:
# If on GPU runtime, use real model; otherwise use mock generator
if HAS_TORCH and torch.cuda.is_available():
    print("Running Day 11 verification with real Qwen2.5-1.5B-Instruct on GPU...")
    !python scripts/test_generation_engine.py --use_real_model --model_name "Qwen/Qwen2.5-1.5B-Instruct"
else:
    print("Running Day 11 verification in mock mode (CPU / local test)...")
    !python scripts/test_generation_engine.py

## Step 7: Day 12 Verification — Evaluation Engine & Metrics on Sample Data

Tests `scripts/evaluate.py` end-to-end:
- Runs greedy evaluation on the 20 sample problems (`val_sample.parquet`)
- Computes math metrics (accuracy, format compliance, fallback rate, token averages)
- Generates `eval_results.json` and `run_info.json` in `outputs/baseline/`

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 12 evaluation on sample data with real Qwen2.5-1.5B on GPU...')
    !python scripts/evaluate.py --stage baseline --preset greedy --split sample --limit 20
else:
    print('Running Day 12 evaluation on sample data in mock mode (CPU / local test)...')
    !python scripts/evaluate.py --stage baseline --preset greedy --split sample --limit 20 --use_mock


## Step 8: Day 13 — Development Model Baseline (500 Validation Problems)

Runs greedy and self-consistency decoding across all 500 validation problems (`val.parquet`):
- Writes to `outputs/baseline/Qwen2.5-1.5B-Instruct/` (one folder per model):
  `eval_results_val_{greedy,self_consistency}.json`, `generations_*.jsonl` (+ `.meta.json`), `run_info.json`
- High-throughput inference via vLLM (or Hugging Face batched generation)

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 13 development baseline on 500 validation problems (GPU)...')
    !python scripts/run_dev_baseline.py --split val
else:
    print('Running Day 13 development baseline in mock mode (CPU / local test)...')
    !python scripts/run_dev_baseline.py --split val --limit 20 --use_mock


## Step 9: Day 13 — Output Audit & Parser Quality Review

Inspects 20 model reasoning completions to verify answer extraction, format compliance, and equivalence checking.
Generates `reports/day13_output_review.md`.

In [ ]:
!python scripts/review_outputs.py --count 20


## Step 10: Day 14 — 7B Baselines (3 Models x 2 Presets on 500 Validation Problems)

Executes baseline evaluations for all three DeepSeek-Math 7B models on `val.parquet` (500 problems):
1. `deepseek-ai/deepseek-math-7b-base` (8-shot few-shot prompt)
2. `deepseek-ai/deepseek-math-7b-instruct` (chat template)
3. `deepseek-ai/deepseek-math-7b-rl` (chat template)

Under `greedy` and `self_consistency` decoding presets.
Outputs 6 result sets in `outputs/baseline/<model_slug>/` and summary table `outputs/baseline/baseline_comparison.md`.

**First G4 run checklist** (vLLM memory release cannot be tested without a GPU):
1. Step 5c ran on this runtime, so vLLM is installed; the log says `Auto-selected generation backend: 'vllm'`.
2. The log shows `prompt_format=few_shot` for the base model and `prompt_format=chat` for instruct and rl.
3. After each run the log prints `GPU memory after release: X GiB free of Y GiB`; X should be close to Y.
   If run 2 fails with CUDA out of memory, run one model per process instead, e.g.
   `!python scripts/run_7b_baselines.py --split val --models deepseek-ai/deepseek-math-7b-base` (then instruct, then rl).
4. In `baseline_comparison.md`, `Truncated (%)` should be near 0; a high value means `max_new_tokens` is too low.
5. A disconnect is safe: rerun the same cell and finished chunks are skipped. Changing the model, prompt or
   decoding settings for an existing folder raises an error instead of mixing runs.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 14 7B baselines on GPU (G4/T4)...')
    !python scripts/run_7b_baselines.py --split val
else:
    print('Running Day 14 7B baselines in mock mode (CPU / local test)...')
    !python scripts/run_7b_baselines.py --split val --limit 20 --use_mock


## Step 11: Day 15 — Safety Prompt Set & Llama Guard Screening

1. Builds `data/safety/` (50 JailbreakBench harmful prompts, 5 per category, + 50 benign GSM8K train questions, seed 42).
2. Runs the safety unit tests.
3. On a GPU with at least 20 GB VRAM (L4 / G4 / A100), generates development-model responses and screens them with `meta-llama/Llama-Guard-3-8B`.
   Llama Guard 3 is gated: `HF_TOKEN` must belong to an account with approved access. A T4 (15 GB) cannot hold it in bf16.

Outputs `outputs/baseline/safety/<model_slug>/safety_results.json`, `safety_assessments.jsonl` and `run_info.json`.

In [ ]:
!python scripts/prepare_safety_data.py
!python -m unittest tests/test_safety.py

if HAS_TORCH and torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if vram_gb >= 20:
        print(f'Running Day 15 safety evaluation on the development model ({vram_gb:.0f} GB GPU)...')
        !python scripts/run_safety_eval.py
    else:
        print(f'GPU has {vram_gb:.0f} GB; Llama Guard 3 8B needs ~16 GB plus headroom. Switch to L4/G4/A100.')
else:
    print('Running Day 15 safety evaluation in mock mode (CPU / local test)...')
    !python scripts/run_safety_eval.py --use_mock --limit 10 --output_dir outputs/mock/safety


## Step 12: Day 15 — Manual Review of 30 Outputs per 7B Model

Writes one review report per model to `reports/day15_review_<model_slug>.md` from the **real** Step 10 greedy generations.
Read every row and check that the parsed answer matches what you would read; add every parser miss to `tests/test_answer_extraction.py` before fixing it.

In [ ]:
from src.config import ConfigManager

_baseline_dir = ConfigManager().get_stage_paths('baseline')['outputs']
for slug in ['deepseek-math-7b-base', 'deepseek-math-7b-instruct', 'deepseek-math-7b-rl']:
    gen_file = _baseline_dir / slug / 'generations_val_greedy.jsonl'
    !python scripts/review_outputs.py --input_file "{gen_file}" --count 30 --output_report reports/day15_review_{slug}.md


## Step 13: Phase 3 Day 16 — Reasoning Prompts Verification (CoT, Solve-then-Verify, Alternative-Method)

Tests all three reasoning prompt strategies on 10 sample problems with the development model:
1. `chain_of_thought`: direct step-by-step mathematical reasoning.
2. `solve_then_verify`: two-stage generation (initial solution followed by verification/sanity check).
3. `alternative_method`: alternative formulation (algebraic modeling, working backwards, or proportional reasoning).

Verifies 100% format compliance with the canonical answer contract (`#### <number>`).
Writes results to `reports/day16_prompt_verification.json` and `reports/day16_reasoning_prompts.md`.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Testing Day 16 reasoning prompts on sample problems with real model on GPU...')
    !python scripts/test_reasoning_prompts.py --use_real_model --num_problems 10
else:
    print('Testing Day 16 reasoning prompts in mock mode (CPU / local)...')
    !python scripts/test_reasoning_prompts.py --num_problems 10


## Step 14: Phase 3 Day 17 — Synthetic Reasoning Trace Sampling (100-Problem Dev Run)

Samples 4 reasoning traces per problem across 100 problems from the 3,000-problem training subset.
Records trace ID, rendered prompt, completion, parsed answer, extraction tier, correctness flag, token counts, and UTC timestamp.
Writes chunked, resumable JSONL to `data/synthetic/generated/` and yield report to `reports/day17_synthetic_generation.md`.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 17 trace sampling with real model on GPU...')
    !python scripts/generate_reasoning_data.py --limit 100 --samples_per_problem 4 --use_real_model
else:
    print('Running Day 17 trace sampling in mock mode (CPU / local)...')
    !python scripts/generate_reasoning_data.py --limit 100 --samples_per_problem 4 --use_mock


## Step 15: Phase 3 Day 18 — 7B Instruct Chain-of-Thought Sampling (3,000 Problems x 4 Samples = 12,000 Traces on G4)

Generates 4 chain-of-thought samples for each of the 3,000 problems from the training subset using `deepseek-ai/deepseek-math-7b-instruct` under the `sampled` preset (temperature 0.7, top-p 0.95, max_new_tokens 512).
Applies the correctness filter (`src/data/curation.py`) to verify answer equivalence and remove truncated or degenerate traces.
Saves chunked JSONL to `data/synthetic/generated/` and logs yield metrics.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 18 7B Instruct Chain-of-Thought sampling on GPU (G4)...')
    !python scripts/generate_reasoning_data.py --model_name "deepseek-ai/deepseek-math-7b-instruct" --samples_per_problem 4 --prompt_types "chain_of_thought" --use_real_model --report_name "day18_cot_sampling"
else:
    print('Running Day 18 CoT sampling in mock mode (CPU / local test)...')
    !python scripts/generate_reasoning_data.py --limit 50 --samples_per_problem 4 --prompt_types "chain_of_thought" --use_mock --report_name "day18_cot_sampling"


## Step 16: Phase 3 Day 19 — Reasoning Trace Yield Analysis & Per-Problem Pass Rates

Computes and saves:
1. Per-problem pass rates (correct samples out of 4) saved to `data/synthetic/generated/problem_pass_rates.json` and `.parquet` (reused in Phase 6 for GRPO difficulty filtering).
2. Overall trace yield, problem solve rate, unsolved count, and formatting failures.
3. Diagnostic markdown report in `reports/day19_yield_analysis.md` and JSON metrics in `reports/day19_yield_analysis_report.json`.
4. Curated SFT dataset of verified correct traces in `data/synthetic/curated/traces_curated_sft.jsonl`.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 19 Yield Analysis on generated 7B CoT traces...')
    !python scripts/analyze_reasoning_yield.py --traces_file data/synthetic/generated/traces_deepseek-math-7b-instruct_chain_of_thought.jsonl --output_dir data/synthetic/generated/ --save_curated
else:
    print('Running Day 19 Yield Analysis in mock/dev mode...')
    !python scripts/analyze_reasoning_yield.py --output_dir data/synthetic/generated/ --save_curated --mock


## Step 17: Phase 3 Day 20 — Verification & Alternative-Method Trace Generation (on Solved CoT Problems)

For every problem with at least one correct chain-of-thought trace (from Day 19 pass rates):
1. Generates 1 `solve_then_verify` trace.
2. Generates 1 `alternative_method` trace.
3. Filters both using `CorrectnessFilter` (`src/data/curation.py`) and slices per-prompt JSONLs for Day 21 judge scoring.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 20 Verification and Alternative-Method trace sampling on GPU (G4)...')
    !python scripts/generate_reasoning_data.py \
        --model_name "deepseek-ai/deepseek-math-7b-instruct" \
        --prompt_types "solve_then_verify,alternative_method" \
        --samples_per_problem 1 \
        --only_solved \
        --use_real_model \
        --report_name "day20_multi_strategy_traces"
else:
    print('Running Day 20 trace sampling in mock/dev mode...')
    !python scripts/generate_reasoning_data.py \
        --limit 50 \
        --prompt_types "solve_then_verify,alternative_method" \
        --samples_per_problem 1 \
        --only_solved \
        --use_mock \
        --report_name "day20_multi_strategy_traces"


## Step 18: Phase 3 Day 21 — LLM Judge Rubric Evaluation & Benchmark Calibration

Evaluates reasoning traces against the 3-dimension rubric (Completeness, Verification Quality, and Novelty).
Runs calibration on the 20 ground-truth benchmark traces in `data/calibration/manual_scores_20.json` to verify Pearson correlation $r \ge 0.60$.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 21 Judge Calibration with model on GPU (G4)...')
    !python scripts/score_traces_judge.py \
        --model_name "deepseek-ai/deepseek-math-7b-instruct" \
        --calibrate \
        --use_real_model \
        --report_name "day21_judge_calibration"
else:
    print('Running Day 21 Judge Calibration in mock/dev mode...')
    !python scripts/score_traces_judge.py \
        --calibrate \
        --use_mock \
        --report_name "day21_judge_calibration"


## Step 19: Phase 3 Day 22 — Full-Scale LLM Judge Scoring & Distribution Analysis (7B Instruct on G4)

Scores all correct reasoning traces across all three prompt strategies (`chain_of_thought`, `solve_then_verify`, `alternative_method`) using `deepseek-ai/deepseek-math-7b-instruct` on GPU (G4).
Strictly enforces Design Rule 4 (correctness filters first, judge ranks second).
Computes calibration metrics on the 20 ground-truth benchmark traces ($r \ge 0.60$), score spread statistics, and assigns within-problem relative ranks (`rank_within_problem`) for Day 23 curation.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 22 Judge Scoring on G4 with deepseek-ai/deepseek-math-7b-instruct...')
    !python scripts/score_traces_judge.py \
        --model_name "deepseek-ai/deepseek-math-7b-instruct" \
        --traces_dir "data/synthetic/generated" \
        --output_file "data/synthetic/generated/traces_scored.jsonl" \
        --split_by_prompt_type \
        --use_real_model \
        --check_calibration \
        --report_name "day22_judge_scoring"
else:
    print('Running Day 22 Judge Scoring in mock/dev mode...')
    !python scripts/score_traces_judge.py \
        --traces_dir "data/synthetic/generated" \
        --output_file "data/synthetic/generated/traces_scored.jsonl" \
        --split_by_prompt_type \
        --use_mock \
        --check_calibration \
        --report_name "day22_judge_scoring"


## Step 20: Phase 3 Day 23 — SFT Reasoning Dataset Curation & Contamination Gate

Executes final SFT reasoning dataset curation from scored synthetic traces and human gold solutions:
- Retains at most 2 traces per problem, prioritizing prompt strategy diversity (e.g. 1 alternative method + 1 CoT) and top judge score.
- Filters near-duplicates (Jaccard $\ge 0.85$), overlength traces (>512 tokens), and degenerate text.
- Uses cleaned human gold solutions for unsolved problems to ensure 100% problem coverage.
- Asserts strict zero contamination against validation and test splits.
- Generates `data/synthetic/curated/traces_curated_sft.jsonl` and diagnostic reports.

In [ ]:
print('Running Day 23 Reasoning Dataset Curation...')
!python scripts/curate_data.py \
    --traces_file "data/synthetic/generated/traces_scored.jsonl" \
    --output_file "data/synthetic/curated/traces_curated_sft.jsonl" \
    --max_traces_per_problem 2 \
    --report_name "day23_curation"


## Step 21: Phase 3 Day 24 — Reasoning Trace Inspection & Systematic Audit

Executes qualitative inspection and audit across curated and rejected reasoning traces:
- Generates 100-trace qualitative audit (50 representative kept + 50 representative rejected traces).
- Analyzes response length distribution percentiles (P25, P50, P75, P95) across all prompt strategies.
- Measures judge score spread and validates within-problem relative ranking.
- Generates `reports/day24_inspection_report.md` and `notebooks/02_inspect_reasoning_traces.ipynb` analysis.

In [ ]:
print('Running Day 24 Trace Inspection & Systematic Audit...')
!python scripts/inspect_reasoning_traces.py \
    --curated_file "data/synthetic/curated/traces_curated_sft.jsonl" \
    --scored_file "data/synthetic/generated/traces_scored.jsonl" \
    --pass_rates_file "data/synthetic/generated/problem_pass_rates.json" \
    --report_name "day24_inspection_report"
